# Dementia detection from the Pitt cookie-theft task

Our goal in this project is to predict Control vs Probable Alzheimer's Disease (AD) from speech segments. We will establish both acoustic and linguistic baselines, predicting at the segment level and aggregating to the session level.

The DementiaBank Pitt Corpus / Cookie Theft task data is available only through authorized course or dataset access. Raw audio, transcripts, and private download links are intentionally not included in this public repository.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

A few Python packages need to be installed to ensure all dependencies are available.

In [ ]:
!pip install transformers datasets torchaudio scikit-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# 0. Loading and exploring the data

DementiaBank's Pitt corpus contains audio and transcripts from people with Probable Alzheimer's Disease (AD) and age-matched controls describing the Boston "Cookie Theft" picture. Picture description is a standard clinical screening task: it elicits spontaneous connected speech and stresses lexical retrieval, coherence, and attention — all of which degrade with AD.

Two modalities are available for each segment (one utterance ≈ a few seconds of PAR speech):
- **Audio** — 16 kHz mono WAV.
- **Text** — cleaned CHAT transcripts.

In [ ]:
SEED = 42

import os
import random
import numpy as np
import torch
from transformers import set_seed

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

In [ ]:
from collections import defaultdict
from pathlib import Path
import torchaudio
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

After downloading the zip file, upload a copy to your Google Drive Account to access it from within the notebook. We will copy it to the local Colab instance and extract it for faster loading.

In [ ]:
!unzip -o "/content/drive/MyDrive/214B_Project2/S26_ECE_214B_Mini_Project_2.zip" -d /content/

PACKAGE_PATH = "/content/S26_ECE_214B_Mini_Project_2"
%cd "$PACKAGE_PATH"

ROOT = Path(".").resolve()
SPLITS_DIR = ROOT / "splits"
SESS_DIR = ROOT / "sessions"
FEATURE_CACHE = ROOT / "features_hubert_base.npz"
MODEL_ID = "facebook/hubert-base-ls960"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

## 1. Load splits (wav + text + labels)

In [ ]:
def load_two_col(path: Path) -> dict[str, str]:
    out: dict[str, str] = {}
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        k, v = line.split("\t", 1)
        out[k] = v
    return out

def load_split(split: str):
    d = SPLITS_DIR / split
    scp = load_two_col(d / "wav.scp")
    text = load_two_col(d / "text")
    lab = {k: int(v) for k, v in load_two_col(d / "utt2label").items()}
    sess = load_two_col(d / "utt2session")
    utts = sorted(scp)
    return {
        "utts": utts,
        "wav": [str(ROOT / scp[u]) for u in utts],
        "text": [text[u] for u in utts],
        "y": [lab[u] for u in utts],
        "sess": [sess[u] for u in utts],
    }

splits = {s: load_split(s) for s in ("train", "dev", "test")}
for s, d in splits.items():
    print(f"{s:5s}: {len(d['utts']):>5d} segments "
          f"({sum(1 for y in d['y'] if y == 0)} Control / "
          f"{sum(1 for y in d['y'] if y == 1)} AD)")

## 2. Shared helpers and baselines


### Segment vs session metrics — why we report both

Both baselines emit one prediction per **segment** (one PAR utterance, ~2–3 s of speech). The classifier's raw output lives at this unit. We then **aggregate to session level** by averaging all segment probabilities within a recording (one `.cha` file). We report metrics at both granularities because they answer different questions:

| | Segment-level | Session-level |
|---|---|---|
| Unit of prediction | one PAR utterance | one full recording |
| Test-set size | 1,191 | 99 |
| How the score is formed | `P(AD ∣ segment)` directly | **mean** of segment probs within the session |
| What it measures | how informative the **embeddings** are | how good the **end-to-end system** is |

A single short utterance is noisy: some Control segments sound AD-like and vice versa,  so segment AUROC is inherently lower. Averaging ~12 segment probabilities per session cancels utterance-level noise; systematic speaker-level signal survives. A large seg→session gap means pooling is doing most of the work; a small gap means the embeddings already carry most of the signal.

Clinical screening acts on a whole recording, not a single utterance, and segment metrics overstate information content because segments inside the same session are heavily correlated. Keep segment metrics for diagnosis/ablation; compare headline results at the session level.

Model selection here is driven by **dev session AUROC**. With test n=99 sessions the 95% CI on AUROC is roughly ±0.08 — treat differences <0.03 between models as noise unless you bootstrap.

In [ ]:
C_GRID = (0.01, 0.1, 0.5, 1.0, 2.0, 4.0, 10.0)


def aggregate_sessions(utts, probs, y, sessions):
    bucket_prob = defaultdict(list)
    bucket_y = {}
    for p, lab, s in zip(probs, y, sessions):
        bucket_prob[s].append(float(p))
        bucket_y[s] = int(lab)
    sids = sorted(bucket_prob)
    sp = np.array([np.mean(bucket_prob[s]) for s in sids])
    sy = np.array([bucket_y[s] for s in sids])
    return sp, sy, sids


def score(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "auroc": float(roc_auc_score(y_true, y_prob)) if len(set(y_true)) == 2 else float("nan"),
        "confusion": confusion_matrix(y_true, y_pred).tolist(),
        "n": int(len(y_true)),
    }


def sweep_logreg(Xtr, ytr, Xdv, dv_u, ydv, sr_dv):
    best = None
    for C in C_GRID:
        clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced")
        clf.fit(Xtr, ytr)
        dv_prob = clf.predict_proba(Xdv)[:, 1]
        sp, sy, _ = aggregate_sessions(dv_u, dv_prob, ydv, sr_dv)
        m = score(sy, sp)
        print(f"  C={C:<5g}  dev session AUROC={m['auroc']:.3f}  F1={m['macro_f1']:.3f}")
        if best is None or m["auroc"] > best[1]:
            best = (C, m["auroc"], clf)
    print(f"  -> best C = {best[0]}")
    return best[2], best[0]


def report(clf, Xtr, ytr, sr_tr, tr_u,
           Xdv, ydv, sr_dv, dv_u,
           Xte, yte, sr_te, te_u):
    results = {}
    for name, X, y, sess, u in (("train", Xtr, ytr, sr_tr, tr_u),
                                ("dev",   Xdv, ydv, sr_dv, dv_u),
                                ("test",  Xte, yte, sr_te, te_u)):
        prob = clf.predict_proba(X)[:, 1]
        seg = score(np.array(y), prob)
        sp, sy, _ = aggregate_sessions(u, prob, y, sess)
        ses = score(sy, sp)
        results[name] = {"segment": seg, "session": ses}
    print(f"{'split':<6} {'seg acc':>8} {'seg F1':>7} {'seg AUROC':>10} "
          f"{'sess acc':>9} {'sess F1':>8} {'sess AUROC':>11}")
    for name in ("train", "dev", "test"):
        r = results[name]
        print(f"{name:<6} {r['segment']['accuracy']:>8.3f} {r['segment']['macro_f1']:>7.3f} "
              f"{r['segment']['auroc']:>10.3f} {r['session']['accuracy']:>9.3f} "
              f"{r['session']['macro_f1']:>8.3f} {r['session']['auroc']:>11.3f}")
    print("Test session confusion [[TN, FP], [FN, TP]]:")
    print(np.array(results["test"]["session"]["confusion"]))
    return results


## 3. Acoustic baseline: HuBERT-base + LogReg

Mean-pool the last hidden state of `facebook/hubert-base-ls960` per segment (768-d) and train a logistic regression probe on top.

In [ ]:
SR = 16000
BATCH = 16


In [ ]:
def _load_wav(path: str) -> torch.Tensor:
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(0, keepdim=True)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    return wav.squeeze(0)


def extract_hubert(pairs):
    """pairs: list of (utt_id, wav_path) -> dict utt_id -> 768-d np.float32."""
    from transformers import AutoFeatureExtractor, AutoModel

    print(f"loading {MODEL_ID} on {device}")
    fe = AutoFeatureExtractor.from_pretrained(MODEL_ID)
    model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()

    feats: dict[str, np.ndarray] = {}
    with torch.inference_mode():
        i = 0
        while i < len(pairs):
            chunk = pairs[i : i + BATCH]
            i += BATCH
            wavs = [_load_wav(p).numpy() for _, p in chunk]
            inputs = fe(wavs, sampling_rate=SR, return_tensors="pt",
                        padding=True, return_attention_mask=True)
            input_values = inputs["input_values"].to(device)
            attn = inputs.get("attention_mask")
            attn = attn.to(device) if attn is not None else None
            out = model(input_values=input_values, attention_mask=attn)
            hidden = out.last_hidden_state
            if attn is not None and hasattr(model, "_get_feat_extract_output_lengths"):
                out_lens = model._get_feat_extract_output_lengths(attn.sum(-1)).to(hidden.device)
                fmask = torch.zeros(hidden.shape[:2], device=hidden.device, dtype=hidden.dtype)
                for b, L in enumerate(out_lens.tolist()):
                    fmask[b, : int(L)] = 1.0
                denom = fmask.sum(1, keepdim=True).clamp(min=1)
                pooled = (hidden * fmask.unsqueeze(-1)).sum(1) / denom
            else:
                pooled = hidden.mean(1)
            for (u, _), e in zip(chunk, pooled.cpu().numpy()):
                feats[u] = e.astype(np.float32)
            if (i // BATCH) % 25 == 0:
                print(f"  extracted ~{min(i, len(pairs))}/{len(pairs)}")
    return feats


if FEATURE_CACHE.exists():
    print(f"loading cached HuBERT features from {FEATURE_CACHE.name}")
    z = np.load(FEATURE_CACHE)
    features = {k: z[k] for k in z.files}
else:
    pairs, seen = [], set()
    for s in ("train", "dev", "test"):
        for u, p in zip(splits[s]["utts"], splits[s]["wav"]):
            if u not in seen:
                pairs.append((u, p))
                seen.add(u)
    features = extract_hubert(pairs)
    np.savez(FEATURE_CACHE, **features)
    print(f"saved {len(features)} features -> {FEATURE_CACHE.name}")


dim = next(iter(features.values())).shape[0]
print(f"feature dim = {dim}, n = {len(features)}")


In [ ]:
def stack_audio(split: str):
    d = splits[split]
    X = np.stack([features[u] for u in d["utts"]]).astype(np.float32)
    return d["utts"], X, np.array(d["y"], dtype=np.int64), d["sess"]


tr_u, Xtr_a, ytr, sr_tr = stack_audio("train")
dv_u, Xdv_a, ydv, sr_dv = stack_audio("dev")
te_u, Xte_a, yte, sr_te = stack_audio("test")

scaler = StandardScaler().fit(Xtr_a)
Xtr_a_s = scaler.transform(Xtr_a)
Xdv_a_s = scaler.transform(Xdv_a)
Xte_a_s = scaler.transform(Xte_a)

print("Sweeping C on dev session AUROC:")
clf_audio, best_C_audio = sweep_logreg(Xtr_a_s, ytr, Xdv_a_s, dv_u, ydv, sr_dv)

print("\n=== Acoustic baseline (HuBERT-base + LogReg) ===")
results_audio = report(
    clf_audio,
    Xtr_a_s, ytr, sr_tr, tr_u,
    Xdv_a_s, ydv, sr_dv, dv_u,
    Xte_a_s, yte, sr_te, te_u,
)


## 4. Linguistic baseline: TF-IDF + LogReg

Word 1- and 2-grams over the raw CHAT transcripts, `sublinear_tf=True`, `min_df=2`.

In [ ]:
# Linguistic baseline: word 1- and 2-gram TF-IDF -> LogReg.
# Same splits, same LogReg sweep recipe, just a different feature space.

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
Xtr_t = vec.fit_transform(splits["train"]["text"])
Xdv_t = vec.transform(splits["dev"]["text"])
Xte_t = vec.transform(splits["test"]["text"])
print(f"TF-IDF features: {Xtr_t.shape[1]}")

print("Sweeping C on dev session AUROC:")
clf_text, best_C_text = sweep_logreg(Xtr_t, ytr, Xdv_t, dv_u, ydv, sr_dv)

print("\n=== Linguistic baseline (TF-IDF + LogReg) ===")
results_text = report(
    clf_text,
    Xtr_t, ytr, sr_tr, tr_u,
    Xdv_t, ydv, sr_dv, dv_u,
    Xte_t, yte, sr_te, te_u,
)

# Peek at the top TF-IDF features the classifier relies on.
coefs = clf_text.coef_[0]
vocab = np.array(vec.get_feature_names_out())
order = np.argsort(coefs)
print("\nTop 15 features pushing toward ProbableAD:")
for i in order[-15:][::-1]:
    print(f"  {vocab[i]:25s}  w={coefs[i]:+.3f}")
print("Top 15 features pushing toward Control:")
for i in order[:15]:
    print(f"  {vocab[i]:25s}  w={coefs[i]:+.3f}")


## 5. Side-by-side test-session comparison

In [ ]:
summary = {
    "HuBERT-base + LogReg (acoustic)": results_audio["test"]["session"],
    "TF-IDF 1-2gram + LogReg (text)":  results_text["test"]["session"],
}
print(f"{'baseline':<36} {'acc':>6} {'macro-F1':>9} {'AUROC':>7}")
for name, m in summary.items():
    print(f"{name:<36} {m['accuracy']:>6.3f} {m['macro_f1']:>9.3f} {m['auroc']:>7.3f}")


## Your tasks

1. Explore other acoustic features, both embedding and non-embedding based (e.g., ecapa-tdnn, wavlm, eGeMAPS, hubert-large).

2. Explore other textual baselines. For example, replace TF-IDF with a pre-trained language model like BERT or RoBERTa.

3. Compare baseline performance across other demographic information (e.g., age, sex).
        
4. For the HuBERT acoustic baseline itself: explore representations from other layers (the baseline uses the last hidden layer).

5. Explore multimodal fusion (combining acoustic and textual features/predictions).

6. **Bonus:** The task is ultimately about **clinical screening**. Propose and implement an operating-point selection procedure that  picks a threshold on dev maximizing F1 subject to recall >0.85 on the AD class, and report test precision/recall at that threshold.

# 1. Dataset audit and sanity checks

Before running additional models, we first inspect the data structure. This helps us understand class balance, speaker/session/segment counts, demographic distributions, transcript length, audio duration, and possible confounds such as age or speech quantity.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

LABEL_NAME = {0: "Control", 1: "ProbableAD"}


def load_session_metadata(sess_dir: Path) -> pd.DataFrame:
    meta = pd.read_csv(sess_dir / "session2meta", sep="\t")

    # age appears as strings like "57;" in this package, so extract the number.
    meta["age_num"] = meta["age"].astype(str).str.extract(r"(\d+)").astype(float)

    session2label = load_two_col(sess_dir / "session2label")
    session2spk = load_two_col(sess_dir / "session2spk")

    meta["label"] = meta["session_id"].map(lambda s: int(session2label[s]))
    meta["label_name"] = meta["label"].map(LABEL_NAME)
    meta["speaker"] = meta["session_id"].map(session2spk)

    return meta


def audio_duration_seconds(path: str) -> float:
    # In newer versions of torchaudio, torchaudio.info() has been removed.
    # We can load the audio to get the waveform and sample rate.
    # torchaudio.load returns a tuple (waveform, sample_rate).
    # waveform has shape (num_channels, num_frames)
    waveform, sample_rate = torchaudio.load(path)
    num_frames = waveform.shape[1]
    return float(num_frames) / float(sample_rate)


def build_utterance_dataframe(splits: dict) -> pd.DataFrame:
    rows = []

    for split, d in splits.items():
        for utt, wav, text, y, session in zip(
            d["utts"], d["wav"], d["text"], d["y"], d["sess"]
        ):
            rows.append({
                "split": split,
                "utt": utt,
                "wav": wav,
                "text": text,
                "label": int(y),
                "label_name": LABEL_NAME[int(y)],
                "session": session,
            })

    utt_df = pd.DataFrame(rows)

    # Recover speaker IDs from the utterance-to-speaker files.
    utt2spk_all = {}
    for split in ("train", "dev", "test"):
        utt2spk_all.update(load_two_col(SPLITS_DIR / split / "utt2spk"))

    utt_df["speaker"] = utt_df["utt"].map(utt2spk_all)

    # Basic text/audio descriptors.
    utt_df["n_words"] = utt_df["text"].astype(str).str.split().map(len)
    utt_df["n_chars"] = utt_df["text"].astype(str).map(len)

    print("Computing audio durations...")
    utt_df["duration_s"] = utt_df["wav"].map(audio_duration_seconds)

    return utt_df


session_meta = load_session_metadata(SESS_DIR)
utt_df = build_utterance_dataframe(splits)

# Map each session to its split.
session_to_split = utt_df.groupby("session")["split"].first()
session_meta["split"] = session_meta["session_id"].map(session_to_split)

# Session-level aggregation.
session_df = (
    utt_df
    .groupby(["split", "session", "speaker", "label", "label_name"], as_index=False)
    .agg(
        n_segments=("utt", "count"),
        total_words=("n_words", "sum"),
        mean_words_per_segment=("n_words", "mean"),
        total_chars=("n_chars", "sum"),
        total_duration_s=("duration_s", "sum"),
        mean_duration_s=("duration_s", "mean"),
    )
)

session_df["speech_rate_wps"] = session_df["total_words"] / session_df["total_duration_s"]

session_df = session_df.merge(
    session_meta[["session_id", "age_num", "sex", "diagnosis", "task"]],
    left_on="session",
    right_on="session_id",
    how="left",
)

print("Utterance dataframe:")
display(utt_df.head())

print("Session dataframe:")
display(session_df.head())

In [ ]:
# -----------------------------
# 1. Split-level summary
# -----------------------------

split_summary = (
    session_df
    .groupby("split")
    .agg(
        speakers=("speaker", "nunique"),
        sessions=("session", "nunique"),
        segments=("n_segments", "sum"),
        control_sessions=("label", lambda x: int((x == 0).sum())),
        ad_sessions=("label", lambda x: int((x == 1).sum())),
    )
    .reset_index()
)

split_summary["control_segments"] = split_summary["split"].map(
    utt_df.groupby("split")["label"].apply(lambda x: int((x == 0).sum()))
)

split_summary["ad_segments"] = split_summary["split"].map(
    utt_df.groupby("split")["label"].apply(lambda x: int((x == 1).sum()))
)

display(split_summary)

In [ ]:
# -----------------------------
# 2. Check speaker-disjoint splits
# -----------------------------

speaker_sets = {
    split: set(utt_df.loc[utt_df["split"] == split, "speaker"])
    for split in ("train", "dev", "test")
}

print("Speaker overlap checks:")
print("train/dev overlap:", len(speaker_sets["train"] & speaker_sets["dev"]))
print("train/test overlap:", len(speaker_sets["train"] & speaker_sets["test"]))
print("dev/test overlap:", len(speaker_sets["dev"] & speaker_sets["test"]))

In [ ]:
# -----------------------------
# 3. Demographic summary
# -----------------------------

demo_summary = (
    session_df
    .groupby(["split", "label_name"])
    .agg(
        n_sessions=("session", "nunique"),
        n_speakers=("speaker", "nunique"),
        mean_age=("age_num", "mean"),
        median_age=("age_num", "median"),
        min_age=("age_num", "min"),
        max_age=("age_num", "max"),
        female=("sex", lambda x: int((x == "female").sum())),
        male=("sex", lambda x: int((x == "male").sum())),
    )
    .reset_index()
)

display(demo_summary)

overall_demo_summary = (
    session_df
    .groupby("label_name")
    .agg(
        n_sessions=("session", "nunique"),
        mean_age=("age_num", "mean"),
        median_age=("age_num", "median"),
        min_age=("age_num", "min"),
        max_age=("age_num", "max"),
        female=("sex", lambda x: int((x == "female").sum())),
        male=("sex", lambda x: int((x == "male").sum())),
    )
    .reset_index()
)

display(overall_demo_summary)

In [ ]:
# -----------------------------
# 4. Sanity plots
# -----------------------------

def plot_box_by_label(df, col, title, ylabel):
    data = [
        df.loc[df["label"] == 0, col].dropna(),
        df.loc[df["label"] == 1, col].dropna(),
    ]

    plt.figure(figsize=(6, 4))
    plt.boxplot(data, labels=["Control", "ProbableAD"])
    plt.title(title)
    plt.ylabel(ylabel)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


plot_box_by_label(
    session_df,
    "age_num",
    "Session age distribution by label",
    "Age"
)

plot_box_by_label(
    session_df,
    "n_segments",
    "Segments per session by label",
    "Number of segments"
)

plot_box_by_label(
    session_df,
    "total_words",
    "Total transcript words per session by label",
    "Total words"
)

plot_box_by_label(
    session_df,
    "total_duration_s",
    "Total audio duration per session by label",
    "Total duration (s)"
)

plot_box_by_label(
    session_df,
    "speech_rate_wps",
    "Speech rate by label",
    "Words per second"
)

In [ ]:
# -----------------------------
# 5. Session-level descriptor table
# -----------------------------

descriptor_cols = [
    "n_segments",
    "total_words",
    "mean_words_per_segment",
    "total_duration_s",
    "mean_duration_s",
    "speech_rate_wps",
    "age_num",
]

descriptor_summary = (
    session_df
    .groupby(["split", "label_name"])[descriptor_cols]
    .agg(["mean", "median", "std"])
)

display(descriptor_summary)

# 2. Simple session-level sanity baselines

We next test how much predictive signal is available from simple, interpretable session-level descriptors before using lexical TF-IDF features or learned acoustic embeddings. This helps determine whether the task can be partially explained by demographic or speech-quantity differences.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
import numpy as np
import pandas as pd

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "AUROC": roc_auc_score(y_true, y_prob),
    }


def run_simple_session_baseline(
    session_df,
    feature_cols,
    categorical_cols=None,
    name="Simple baseline",
):
    if categorical_cols is None:
        categorical_cols = []

    numeric_cols = [c for c in feature_cols if c not in categorical_cols]

    train_df = session_df[session_df["split"] == "train"].copy()
    dev_df = session_df[session_df["split"] == "dev"].copy()
    test_df = session_df[session_df["split"] == "test"].copy()

    X_train = train_df[feature_cols]
    y_train = train_df["label"].values

    X_dev = dev_df[feature_cols]
    y_dev = dev_df["label"].values

    X_test = test_df[feature_cols]
    y_test = test_df["label"].values

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ],
        remainder="drop",
    )

    clf = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("logreg", LogisticRegression(max_iter=5000, class_weight="balanced")),
        ]
    )

    clf.fit(X_train, y_train)

    dev_prob = clf.predict_proba(X_dev)[:, 1]
    test_prob = clf.predict_proba(X_test)[:, 1]

    dev_metrics = compute_metrics(y_dev, dev_prob)
    test_metrics = compute_metrics(y_test, test_prob)

    rows = []
    for split_name, metrics in [("dev", dev_metrics), ("test", test_metrics)]:
        row = {"Model": name, "Split": split_name}
        row.update(metrics)
        rows.append(row)

    return clf, pd.DataFrame(rows), {
        "dev_prob": dev_prob,
        "test_prob": test_prob,
        "dev_y": y_dev,
        "test_y": y_test,
        "dev_sessions": dev_df["session"].values,
        "test_sessions": test_df["session"].values,
    }

In [ ]:
simple_baseline_results = []

# 1. Demographics only
demo_features = ["age_num", "sex"]

demo_model, demo_results, demo_outputs = run_simple_session_baseline(
    session_df=session_df,
    feature_cols=demo_features,
    categorical_cols=["sex"],
    name="Demographics only",
)

simple_baseline_results.append(demo_results)


# 2. Speech quantity only
quantity_features = [
    "n_segments",
    "total_words",
    "mean_words_per_segment",
    "total_duration_s",
    "mean_duration_s",
    "speech_rate_wps",
]

quantity_model, quantity_results, quantity_outputs = run_simple_session_baseline(
    session_df=session_df,
    feature_cols=quantity_features,
    categorical_cols=[],
    name="Speech quantity only",
)

simple_baseline_results.append(quantity_results)


# 3. Demographics + speech quantity
all_simple_features = demo_features + quantity_features

simple_model, simple_results, simple_outputs = run_simple_session_baseline(
    session_df=session_df,
    feature_cols=all_simple_features,
    categorical_cols=["sex"],
    name="Demographics + speech quantity",
)

simple_baseline_results.append(simple_results)


simple_baseline_table = pd.concat(simple_baseline_results, ignore_index=True)

for col in ["Accuracy", "F1", "Precision", "Recall", "AUROC"]:
    simple_baseline_table[col] = simple_baseline_table[col].map(lambda x: round(100 * x, 2))

display(simple_baseline_table)

In [ ]:
extra_demo_results = []

age_model, age_results, age_outputs = run_simple_session_baseline(
    session_df=session_df,
    feature_cols=["age_num"],
    categorical_cols=[],
    name="Age only",
)
extra_demo_results.append(age_results)

sex_model, sex_results, sex_outputs = run_simple_session_baseline(
    session_df=session_df,
    feature_cols=["sex"],
    categorical_cols=["sex"],
    name="Sex only",
)
extra_demo_results.append(sex_results)

age_sex_model, age_sex_results, age_sex_outputs = run_simple_session_baseline(
    session_df=session_df,
    feature_cols=["age_num", "sex"],
    categorical_cols=["sex"],
    name="Age + sex",
)
extra_demo_results.append(age_sex_results)

extra_demo_table = pd.concat(extra_demo_results, ignore_index=True)

for col in ["Accuracy", "F1", "Precision", "Recall", "AUROC"]:
    extra_demo_table[col] = extra_demo_table[col].map(lambda x: round(100 * x, 2))

display(extra_demo_table)

The model may be using age as a shortcut. Since AD speakers are older in train and test, the test score might look better than it would on a fair age-balanced dataset. So we need to be careful and check whether the model is learning dementia speech patterns or just age-related patterns.

# 3. Baseline error analysis

We next inspect the session-level predictions from the acoustic and linguistic baselines. This helps determine whether the two modalities make similar errors, whether errors correlate with age, and whether multimodal fusion is likely to help.

In [ ]:
def session_predictions_from_model(clf, X, y, sessions, utts, model_name):
    """
    Convert segment-level model probabilities into session-level predictions.
    """
    seg_prob = clf.predict_proba(X)[:, 1]
    sess_prob, sess_y, sess_ids = aggregate_sessions(utts, seg_prob, y, sessions)

    out = pd.DataFrame({
        "session": sess_ids,
        f"{model_name}_prob": sess_prob,
        "label": sess_y,
    })

    out[f"{model_name}_pred"] = (out[f"{model_name}_prob"] >= 0.5).astype(int)
    out[f"{model_name}_correct"] = out[f"{model_name}_pred"] == out["label"]

    return out


hubert_test_pred = session_predictions_from_model(
    clf_audio,
    Xte_a_s,
    yte,
    sr_te,
    te_u,
    model_name="hubert",
)

tfidf_test_pred = session_predictions_from_model(
    clf_text,
    Xte_t,
    yte,
    sr_te,
    te_u,
    model_name="tfidf",
)

baseline_error_df = hubert_test_pred.merge(
    tfidf_test_pred.drop(columns=["label"]),
    on="session",
    how="inner",
)

baseline_error_df = baseline_error_df.merge(
    session_df[
        [
            "session",
            "label_name",
            "age_num",
            "sex",
            "n_segments",
            "total_words",
            "total_duration_s",
            "speech_rate_wps",
        ]
    ],
    on="session",
    how="left",
)

baseline_error_df["both_correct"] = (
    baseline_error_df["hubert_correct"] & baseline_error_df["tfidf_correct"]
)

baseline_error_df["both_wrong"] = (
    ~baseline_error_df["hubert_correct"] & ~baseline_error_df["tfidf_correct"]
)

baseline_error_df["hubert_only_correct"] = (
    baseline_error_df["hubert_correct"] & ~baseline_error_df["tfidf_correct"]
)

baseline_error_df["tfidf_only_correct"] = (
    ~baseline_error_df["hubert_correct"] & baseline_error_df["tfidf_correct"]
)

display(baseline_error_df.head())

In [ ]:
error_overlap_summary = pd.DataFrame({
    "Category": [
        "Both correct",
        "Both wrong",
        "HuBERT correct, TF-IDF wrong",
        "TF-IDF correct, HuBERT wrong",
    ],
    "Sessions": [
        int(baseline_error_df["both_correct"].sum()),
        int(baseline_error_df["both_wrong"].sum()),
        int(baseline_error_df["hubert_only_correct"].sum()),
        int(baseline_error_df["tfidf_only_correct"].sum()),
    ],
})

error_overlap_summary["Percent"] = (
    100 * error_overlap_summary["Sessions"] / len(baseline_error_df)
).round(2)

display(error_overlap_summary)

In [ ]:
def summarize_errors_by_model(df, model_name):
    correct_col = f"{model_name}_correct"
    pred_col = f"{model_name}_pred"
    prob_col = f"{model_name}_prob"

    summary = (
        df
        .assign(error_type=lambda x: np.select(
            [
                (x["label"] == 0) & (x[pred_col] == 1),
                (x["label"] == 1) & (x[pred_col] == 0),
                x[correct_col],
            ],
            [
                "False positive: Control predicted AD",
                "False negative: AD predicted Control",
                "Correct",
            ],
            default="Other",
        ))
        .groupby("error_type")
        .agg(
            n_sessions=("session", "count"),
            mean_age=("age_num", "mean"),
            median_age=("age_num", "median"),
            mean_total_words=("total_words", "mean"),
            mean_duration_s=("total_duration_s", "mean"),
            mean_prob_AD=(prob_col, "mean"),
        )
        .reset_index()
    )

    return summary


print("HuBERT error summary:")
display(summarize_errors_by_model(baseline_error_df, "hubert"))

print("TF-IDF error summary:")
display(summarize_errors_by_model(baseline_error_df, "tfidf"))

In [ ]:
plt.figure(figsize=(6, 5))

for label_value, label_text in [(0, "Control"), (1, "ProbableAD")]:
    sub = baseline_error_df[baseline_error_df["label"] == label_value]
    plt.scatter(
        sub["tfidf_prob"],
        sub["hubert_prob"],
        alpha=0.7,
        label=label_text,
    )

plt.axvline(0.5, linestyle="--", linewidth=1)
plt.axhline(0.5, linestyle="--", linewidth=1)
plt.xlabel("TF-IDF session probability of AD")
plt.ylabel("HuBERT session probability of AD")
plt.title("HuBERT vs TF-IDF session-level predictions")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 4. Multimodal late fusion

The error analysis showed that TF-IDF is the stronger modality, but HuBERT correctly classifies some sessions that TF-IDF misses. This suggests that acoustic features may provide complementary information. We therefore test simple late-fusion methods using session-level probabilities from the text and acoustic baselines.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
import numpy as np
import pandas as pd

def eval_session_probs(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "AUROC": roc_auc_score(y_true, y_prob),
    }


def make_session_pred_df(clf, X, y, sessions, utts, model_name):
    seg_prob = clf.predict_proba(X)[:, 1]
    sess_prob, sess_y, sess_ids = aggregate_sessions(utts, seg_prob, y, sessions)

    return pd.DataFrame({
        "session": sess_ids,
        "label": sess_y,
        f"{model_name}_prob": sess_prob,
    })


# Dev session probabilities
hubert_dev_pred = make_session_pred_df(
    clf_audio,
    Xdv_a_s,
    ydv,
    sr_dv,
    dv_u,
    "hubert",
)

tfidf_dev_pred = make_session_pred_df(
    clf_text,
    Xdv_t,
    ydv,
    sr_dv,
    dv_u,
    "tfidf",
)

dev_fusion_df = hubert_dev_pred.merge(
    tfidf_dev_pred.drop(columns=["label"]),
    on="session",
    how="inner",
)


# Test session probabilities
hubert_test_pred = make_session_pred_df(
    clf_audio,
    Xte_a_s,
    yte,
    sr_te,
    te_u,
    "hubert",
)

tfidf_test_pred = make_session_pred_df(
    clf_text,
    Xte_t,
    yte,
    sr_te,
    te_u,
    "tfidf",
)

test_fusion_df = hubert_test_pred.merge(
    tfidf_test_pred.drop(columns=["label"]),
    on="session",
    how="inner",
)

display(dev_fusion_df.head())
display(test_fusion_df.head())

In [ ]:
fusion_rows = []

# Baseline individual models
for split_name, df in [("dev", dev_fusion_df), ("test", test_fusion_df)]:
    y = df["label"].values

    for model_name, prob_col in [
        ("HuBERT only", "hubert_prob"),
        ("TF-IDF only", "tfidf_prob"),
    ]:
        metrics = eval_session_probs(y, df[prob_col].values)
        row = {
            "Model": model_name,
            "Split": split_name,
            "alpha_tfidf": np.nan,
        }
        row.update(metrics)
        fusion_rows.append(row)


# Simple average fusion
for split_name, df in [("dev", dev_fusion_df), ("test", test_fusion_df)]:
    y = df["label"].values
    fused_prob = 0.5 * df["tfidf_prob"].values + 0.5 * df["hubert_prob"].values

    metrics = eval_session_probs(y, fused_prob)
    row = {
        "Model": "Average fusion",
        "Split": split_name,
        "alpha_tfidf": 0.5,
    }
    row.update(metrics)
    fusion_rows.append(row)


# Tune alpha on dev AUROC
alphas = np.linspace(0, 1, 21)

alpha_search = []

for alpha in alphas:
    dev_y = dev_fusion_df["label"].values

    dev_fused_prob = (
        alpha * dev_fusion_df["tfidf_prob"].values
        + (1 - alpha) * dev_fusion_df["hubert_prob"].values
    )

    metrics = eval_session_probs(dev_y, dev_fused_prob)

    alpha_search.append({
        "alpha_tfidf": alpha,
        "dev_AUROC": metrics["AUROC"],
        "dev_F1": metrics["F1"],
        "dev_Accuracy": metrics["Accuracy"],
    })

alpha_search_df = pd.DataFrame(alpha_search)

display(alpha_search_df)

best_alpha = alpha_search_df.sort_values(
    ["dev_AUROC", "dev_F1"],
    ascending=False,
).iloc[0]["alpha_tfidf"]

print(f"Best alpha on dev: {best_alpha:.2f}")

In [ ]:
# Evaluate tuned weighted fusion on dev and test
for split_name, df in [("dev", dev_fusion_df), ("test", test_fusion_df)]:
    y = df["label"].values

    fused_prob = (
        best_alpha * df["tfidf_prob"].values
        + (1 - best_alpha) * df["hubert_prob"].values
    )

    metrics = eval_session_probs(y, fused_prob)

    row = {
        "Model": "Weighted fusion tuned on dev",
        "Split": split_name,
        "alpha_tfidf": best_alpha,
    }
    row.update(metrics)
    fusion_rows.append(row)


fusion_results = pd.DataFrame(fusion_rows)

for col in ["Accuracy", "F1", "Precision", "Recall", "AUROC"]:
    fusion_results[col] = fusion_results[col].map(lambda x: round(100 * x, 2))

fusion_results["alpha_tfidf"] = fusion_results["alpha_tfidf"].map(
    lambda x: "" if pd.isna(x) else round(float(x), 2)
)

display(fusion_results)

# 5. Clinical operating-point selection

For clinical screening, missing ProbableAD cases is costly. Instead of using the default 0.5 threshold, we select a decision threshold on the dev set that maximizes F1 while requiring AD recall to be at least 0.85. We then apply that fixed threshold to the test set.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
import numpy as np
import pandas as pd

def threshold_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "threshold": threshold,
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "AUROC": roc_auc_score(y_true, y_prob),
    }


def select_threshold_on_dev(y_dev, p_dev, min_recall=0.85):
    """
    Select threshold on dev that maximizes F1 subject to AD recall >= min_recall.
    """
    thresholds = np.linspace(0.0, 1.0, 1001)
    rows = []

    for thr in thresholds:
        m = threshold_metrics(y_dev, p_dev, thr)
        rows.append(m)

    search_df = pd.DataFrame(rows)
    feasible = search_df[search_df["Recall"] >= min_recall].copy()

    if len(feasible) == 0:
        raise ValueError(f"No threshold satisfies recall >= {min_recall}")

    best = feasible.sort_values(
        ["F1", "Precision", "Accuracy"],
        ascending=False,
    ).iloc[0]

    return float(best["threshold"]), search_df, best


def evaluate_clinical_threshold(name, y_dev, p_dev, y_test, p_test, min_recall=0.85):
    best_thr, search_df, dev_best = select_threshold_on_dev(
        y_dev,
        p_dev,
        min_recall=min_recall,
    )

    dev_metrics = threshold_metrics(y_dev, p_dev, best_thr)
    test_metrics = threshold_metrics(y_test, p_test, best_thr)

    rows = []
    for split_name, metrics in [("dev", dev_metrics), ("test", test_metrics)]:
        row = {
            "Model": name,
            "Split": split_name,
            "Selected threshold": best_thr,
        }
        row.update(metrics)
        rows.append(row)

    return pd.DataFrame(rows), search_df

In [ ]:
# Prepare probabilities
dev_y = dev_fusion_df["label"].values
test_y = test_fusion_df["label"].values

dev_tfidf_prob = dev_fusion_df["tfidf_prob"].values
test_tfidf_prob = test_fusion_df["tfidf_prob"].values

dev_weighted_fusion_prob = (
    best_alpha * dev_fusion_df["tfidf_prob"].values
    + (1 - best_alpha) * dev_fusion_df["hubert_prob"].values
)

test_weighted_fusion_prob = (
    best_alpha * test_fusion_df["tfidf_prob"].values
    + (1 - best_alpha) * test_fusion_df["hubert_prob"].values
)

clinical_rows = []

tfidf_clinical_results, tfidf_threshold_search = evaluate_clinical_threshold(
    name="TF-IDF clinical threshold",
    y_dev=dev_y,
    p_dev=dev_tfidf_prob,
    y_test=test_y,
    p_test=test_tfidf_prob,
    min_recall=0.85,
)

fusion_clinical_results, fusion_threshold_search = evaluate_clinical_threshold(
    name="Weighted fusion clinical threshold",
    y_dev=dev_y,
    p_dev=dev_weighted_fusion_prob,
    y_test=test_y,
    p_test=test_weighted_fusion_prob,
    min_recall=0.85,
)

clinical_results = pd.concat(
    [tfidf_clinical_results, fusion_clinical_results],
    ignore_index=True,
)

clinical_results_display = clinical_results.copy()

for col in ["Accuracy", "F1", "Precision", "Recall", "AUROC"]:
    clinical_results_display[col] = clinical_results_display[col].map(lambda x: round(100 * x, 2))

clinical_results_display["Selected threshold"] = clinical_results_display["Selected threshold"].map(
    lambda x: round(float(x), 3)
)

display(clinical_results_display)

In [ ]:
def plot_threshold_search(search_df, title):
    plt.figure(figsize=(7, 4))
    plt.plot(search_df["threshold"], search_df["Precision"], label="Precision")
    plt.plot(search_df["threshold"], search_df["Recall"], label="Recall")
    plt.plot(search_df["threshold"], search_df["F1"], label="F1")
    plt.axhline(0.85, linestyle="--", linewidth=1, label="Recall = 0.85")
    plt.xlabel("Decision threshold")
    plt.ylabel("Metric")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


plot_threshold_search(
    tfidf_threshold_search,
    "TF-IDF dev threshold selection"
)

plot_threshold_search(
    fusion_threshold_search,
    "Weighted fusion dev threshold selection"
)

In [ ]:
recall_targets = [0.85, 0.90, 0.95, 1.00]

recall_sweep_rows = []

for min_recall in recall_targets:
    for name, p_dev, p_test in [
        ("TF-IDF", dev_tfidf_prob, test_tfidf_prob),
        ("Weighted fusion", dev_weighted_fusion_prob, test_weighted_fusion_prob),
    ]:
        try:
            best_thr, _, _ = select_threshold_on_dev(
                dev_y,
                p_dev,
                min_recall=min_recall,
            )

            dev_metrics = threshold_metrics(dev_y, p_dev, best_thr)
            test_metrics = threshold_metrics(test_y, p_test, best_thr)

            row = {
                "Model": name,
                "Dev recall constraint": min_recall,
                "Selected threshold": best_thr,
                "Dev Precision": dev_metrics["Precision"],
                "Dev Recall": dev_metrics["Recall"],
                "Dev F1": dev_metrics["F1"],
                "Test Precision": test_metrics["Precision"],
                "Test Recall": test_metrics["Recall"],
                "Test F1": test_metrics["F1"],
                "Test Accuracy": test_metrics["Accuracy"],
                "Test AUROC": test_metrics["AUROC"],
            }

            recall_sweep_rows.append(row)

        except ValueError:
            print(f"No feasible threshold for {name} with recall >= {min_recall}")

recall_sweep_df = pd.DataFrame(recall_sweep_rows)

display_df = recall_sweep_df.copy()

for col in [
    "Dev recall constraint",
    "Selected threshold",
    "Dev Precision",
    "Dev Recall",
    "Dev F1",
    "Test Precision",
    "Test Recall",
    "Test F1",
    "Test Accuracy",
    "Test AUROC",
]:
    display_df[col] = display_df[col].map(lambda x: round(float(x), 3))

display(display_df)

# 6. Age-stratified performance analysis

Because the data audit showed that ProbableAD sessions are older than Control sessions on average, we evaluate model performance across age groups. This helps determine whether the models behave consistently across younger and older speakers.

In [ ]:
# Merge model probabilities with session metadata
age_perf_df = test_fusion_df.merge(
    session_df[
        [
            "session",
            "age_num",
            "sex",
            "label_name",
            "n_segments",
            "total_words",
            "total_duration_s",
            "speech_rate_wps",
        ]
    ],
    on="session",
    how="left",
)

# Add weighted fusion probability
age_perf_df["weighted_fusion_prob"] = (
    best_alpha * age_perf_df["tfidf_prob"]
    + (1 - best_alpha) * age_perf_df["hubert_prob"]
)

# Age bins
age_perf_df["age_bin"] = pd.cut(
    age_perf_df["age_num"],
    bins=[0, 65, 75, 200],
    labels=["<65", "65-74", "75+"],
    right=False,
)

display(age_perf_df.head())
display(age_perf_df["age_bin"].value_counts().sort_index())

In [ ]:
age_bin_counts = (
    age_perf_df
    .groupby(["age_bin", "label_name"], observed=False)
    .size()
    .reset_index(name="n_sessions")
)

display(age_bin_counts)

age_bin_pivot = age_bin_counts.pivot(
    index="age_bin",
    columns="label_name",
    values="n_sessions",
).fillna(0).astype(int)

display(age_bin_pivot)

In [ ]:
def safe_metrics_by_group(df, prob_col, model_name, threshold=0.5):
    rows = []

    for age_bin, sub in df.groupby("age_bin", observed=False):
        if len(sub) == 0:
            continue

        y_true = sub["label"].values
        y_prob = sub[prob_col].values
        y_pred = (y_prob >= threshold).astype(int)

        # AUROC is undefined if only one class is present.
        if len(np.unique(y_true)) == 2:
            auroc = roc_auc_score(y_true, y_prob)
        else:
            auroc = np.nan

        rows.append({
            "Model": model_name,
            "Age bin": age_bin,
            "n_sessions": len(sub),
            "n_control": int((y_true == 0).sum()),
            "n_AD": int((y_true == 1).sum()),
            "Accuracy": accuracy_score(y_true, y_pred),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "AUROC": auroc,
        })

    return pd.DataFrame(rows)


age_stratified_results = pd.concat(
    [
        safe_metrics_by_group(age_perf_df, "hubert_prob", "HuBERT"),
        safe_metrics_by_group(age_perf_df, "tfidf_prob", "TF-IDF"),
        safe_metrics_by_group(age_perf_df, "weighted_fusion_prob", "Weighted fusion"),
    ],
    ignore_index=True,
)

age_stratified_display = age_stratified_results.copy()

for col in ["Accuracy", "F1", "Precision", "Recall", "AUROC"]:
    age_stratified_display[col] = age_stratified_display[col].map(
        lambda x: "" if pd.isna(x) else round(100 * x, 2)
    )

display(age_stratified_display)

In [ ]:
plot_df = age_stratified_results.copy()

plt.figure(figsize=(7, 4))

for model_name in ["HuBERT", "TF-IDF", "Weighted fusion"]:
    sub = plot_df[plot_df["Model"] == model_name]
    plt.plot(
        sub["Age bin"].astype(str),
        100 * sub["AUROC"],
        marker="o",
        label=model_name,
    )

plt.xlabel("Age group")
plt.ylabel("AUROC")
plt.title("Test AUROC by age group")
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 7. TF-IDF feature inspection

Since the TF-IDF linguistic baseline was the strongest individual model, we inspect the learned logistic regression coefficients. Positive coefficients indicate words or bigrams that push the model toward ProbableAD, while negative coefficients push the model toward Control. This helps determine whether the model is using plausible linguistic cues or possible dataset artifacts.

In [ ]:
# Inspect TF-IDF logistic regression coefficients

def inspect_tfidf_coefficients(vec_model, logreg_model, top_k=30):
    """
    Extract top positive and negative TF-IDF features from a trained
    TF-IDF Vectorizer and LogisticRegression model.

    Positive coefficients -> push toward ProbableAD
    Negative coefficients -> push toward Control
    """

    # The models are directly provided as arguments
    tfidf_step = vec_model
    logreg_step = logreg_model

    feature_names = np.array(tfidf_step.get_feature_names_out())
    coefs = logreg_step.coef_[0]

    coef_df = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefs,
        "abs_coefficient": np.abs(coefs),
    })

    top_ad = (
        coef_df
        .sort_values("coefficient", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    top_control = (
        coef_df
        .sort_values("coefficient", ascending=True)
        .head(top_k)
        .reset_index(drop=True)
    )

    return coef_df, top_ad, top_control


tfidf_coef_df, top_ad_terms, top_control_terms = inspect_tfidf_coefficients(
    vec, # Pass the TfidfVectorizer object directly
    clf_text, # Pass the LogisticRegression model directly
    top_k=30,
)

print("Top TF-IDF features pushing toward ProbableAD:")
display(top_ad_terms[["feature", "coefficient"]])

print("Top TF-IDF features pushing toward Control:")
display(top_control_terms[["feature", "coefficient"]])

In [ ]:
top_k_display = 15

tfidf_interpretability_table = pd.DataFrame({
    "ProbableAD-associated terms": top_ad_terms["feature"].head(top_k_display).values,
    "AD coefficient": top_ad_terms["coefficient"].head(top_k_display).round(3).values,
    "Control-associated terms": top_control_terms["feature"].head(top_k_display).values,
    "Control coefficient": top_control_terms["coefficient"].head(top_k_display).round(3).values,
})

display(tfidf_interpretability_table)

In [ ]:
# Plot top positive and negative terms

plot_top_k = 15

plot_df = pd.concat([
    top_control_terms.head(plot_top_k).assign(direction="Control"),
    top_ad_terms.head(plot_top_k).assign(direction="ProbableAD"),
])

# Sort so the bars appear cleanly
plot_df = plot_df.sort_values("coefficient")

plt.figure(figsize=(8, 8))
plt.barh(plot_df["feature"], plot_df["coefficient"])
plt.axvline(0, linewidth=1)
plt.xlabel("Logistic regression coefficient")
plt.title("Top TF-IDF features associated with Control vs ProbableAD")
plt.grid(axis="x", alpha=0.3)
plt.show()

In [ ]:
tfidf_coef_df["ngram_len"] = tfidf_coef_df["feature"].str.split().map(len)

ngram_summary = (
    tfidf_coef_df
    .assign(abs_coef=lambda x: x["coefficient"].abs())
    .groupby("ngram_len")
    .agg(
        n_features=("feature", "count"),
        mean_abs_coef=("abs_coef", "mean"),
        max_abs_coef=("abs_coef", "max"),
    )
    .reset_index()
)

display(ngram_summary)

print("Top AD-associated unigrams:")
display(
    tfidf_coef_df[tfidf_coef_df["ngram_len"] == 1]
    .sort_values("coefficient", ascending=False)
    .head(15)[["feature", "coefficient"]]
)

print("Top AD-associated bigrams:")
display(
    tfidf_coef_df[tfidf_coef_df["ngram_len"] == 2]
    .sort_values("coefficient", ascending=False)
    .head(15)[["feature", "coefficient"]]
)

print("Top Control-associated unigrams:")
display(
    tfidf_coef_df[tfidf_coef_df["ngram_len"] == 1]
    .sort_values("coefficient", ascending=True)
    .head(15)[["feature", "coefficient"]]
)

print("Top Control-associated bigrams:")
display(
    tfidf_coef_df[tfidf_coef_df["ngram_len"] == 2]
    .sort_values("coefficient", ascending=True)
    .head(15)[["feature", "coefficient"]]
)

In [ ]:
xxx_summary = (
    session_df
    .assign(
        has_xxx=lambda x: x["session"].map(
            utt_df.groupby("session")["text"].apply(
                lambda texts: any("xxx" in str(t).lower().split() for t in texts)
            )
        )
    )
    .groupby(["split", "label_name"])
    .agg(
        n_sessions=("session", "count"),
        sessions_with_xxx=("has_xxx", "sum"),
        pct_with_xxx=("has_xxx", "mean"),
    )
    .reset_index()
)

xxx_summary["pct_with_xxx"] = (100 * xxx_summary["pct_with_xxx"]).round(2)

display(xxx_summary)

# 8. Transcript-level error examples

To better understand the TF-IDF model's decisions, we inspect short transcript excerpts from correctly and incorrectly classified test sessions. This helps connect quantitative errors to qualitative speech/language patterns such as specificity, repetition, vague references, and unclear-speech markers.

In [ ]:
# Build full session-level transcript text
session_text_df = (
    utt_df
    .groupby("session")
    .agg(
        full_text=("text", lambda xs: " ".join(str(x) for x in xs)),
        n_utterances=("utt", "count"),
    )
    .reset_index()
)

# Add simple text markers
session_text_df["n_words_full"] = session_text_df["full_text"].str.split().map(len)
session_text_df["has_xxx"] = session_text_df["full_text"].str.lower().str.contains(r"\bxxx\b", regex=True)
session_text_df["n_xxx"] = session_text_df["full_text"].str.lower().str.count(r"\bxxx\b")

# Start from test predictions
tfidf_error_examples_df = test_fusion_df.copy()
tfidf_error_examples_df["tfidf_pred"] = (tfidf_error_examples_df["tfidf_prob"] >= 0.5).astype(int)
tfidf_error_examples_df["tfidf_correct"] = tfidf_error_examples_df["tfidf_pred"] == tfidf_error_examples_df["label"]

tfidf_error_examples_df = tfidf_error_examples_df.merge(
    session_df[
        [
            "session",
            "label_name",
            "age_num",
            "sex",
            "n_segments",
            "total_words",
            "total_duration_s",
            "speech_rate_wps",
        ]
    ],
    on="session",
    how="left",
)

tfidf_error_examples_df = tfidf_error_examples_df.merge(
    session_text_df,
    on="session",
    how="left",
)

tfidf_error_examples_df["error_type"] = np.select(
    [
        (tfidf_error_examples_df["label"] == 0) & (tfidf_error_examples_df["tfidf_pred"] == 1),
        (tfidf_error_examples_df["label"] == 1) & (tfidf_error_examples_df["tfidf_pred"] == 0),
        (tfidf_error_examples_df["label"] == 0) & (tfidf_error_examples_df["tfidf_pred"] == 0),
        (tfidf_error_examples_df["label"] == 1) & (tfidf_error_examples_df["tfidf_pred"] == 1),
    ],
    [
        "False positive: Control predicted AD",
        "False negative: AD predicted Control",
        "True negative: Control predicted Control",
        "True positive: AD predicted AD",
    ],
    default="Other",
)

display(
    tfidf_error_examples_df[
        [
            "session",
            "label_name",
            "tfidf_prob",
            "tfidf_pred",
            "error_type",
            "age_num",
            "sex",
            "total_words",
            "has_xxx",
            "n_xxx",
        ]
    ].head()
)

In [ ]:
def get_examples_by_error_type(df, error_type, n=3):
    sub = df[df["error_type"] == error_type].copy()

    if len(sub) == 0:
        return sub

    # For errors, sort by model confidence in the wrong direction.
    if "False positive" in error_type:
        sub = sub.sort_values("tfidf_prob", ascending=False)
    elif "False negative" in error_type:
        sub = sub.sort_values("tfidf_prob", ascending=True)
    elif "True positive" in error_type:
        sub = sub.sort_values("tfidf_prob", ascending=False)
    elif "True negative" in error_type:
        sub = sub.sort_values("tfidf_prob", ascending=True)

    return sub.head(n)


example_sets = []

for err_type in [
    "False positive: Control predicted AD",
    "False negative: AD predicted Control",
    "True positive: AD predicted AD",
    "True negative: Control predicted Control",
]:
    ex = get_examples_by_error_type(tfidf_error_examples_df, err_type, n=3)
    example_sets.append(ex)

selected_examples_df = pd.concat(example_sets, ignore_index=True)

display(
    selected_examples_df[
        [
            "session",
            "error_type",
            "label_name",
            "tfidf_prob",
            "age_num",
            "sex",
            "total_words",
            "n_xxx",
            "full_text",
        ]
    ]
)

In [ ]:
def print_transcript_examples(df, max_chars=700):
    for _, row in df.iterrows():
        print("=" * 100)
        print(f"Session: {row['session']}")
        print(f"Type: {row['error_type']}")
        print(f"True label: {row['label_name']}")
        print(f"TF-IDF probability of AD: {row['tfidf_prob']:.3f}")
        print(f"Age: {row['age_num']}, Sex: {row['sex']}")
        print(f"Total words: {row['total_words']}, xxx count: {row['n_xxx']}")
        print("-" * 100)

        text = row["full_text"]
        if len(text) > max_chars:
            text = text[:max_chars] + " ..."

        print(text)
        print()

print_transcript_examples(selected_examples_df, max_chars=700)

In [ ]:
error_marker_summary = (
    tfidf_error_examples_df
    .groupby("error_type")
    .agg(
        n_sessions=("session", "count"),
        mean_prob_AD=("tfidf_prob", "mean"),
        mean_age=("age_num", "mean"),
        mean_total_words=("total_words", "mean"),
        pct_with_xxx=("has_xxx", "mean"),
        mean_n_xxx=("n_xxx", "mean"),
    )
    .reset_index()
)

error_marker_summary["pct_with_xxx"] = 100 * error_marker_summary["pct_with_xxx"]

display(error_marker_summary.round(3))

# 9. Clinical threshold tradeoff

The initial dev-selected threshold did not preserve high AD recall on test. We therefore sweep stricter dev recall constraints and visualize the resulting test precision-recall tradeoff. This shows how conservative thresholding can improve sensitivity at the cost of more false positives.

In [ ]:
# Clean display of recall-sweep results

clinical_tradeoff_df = recall_sweep_df.copy()

clinical_tradeoff_display = clinical_tradeoff_df[
    [
        "Model",
        "Dev recall constraint",
        "Selected threshold",
        "Test Precision",
        "Test Recall",
        "Test F1",
        "Test Accuracy",
        "Test AUROC",
    ]
].copy()

display(clinical_tradeoff_display.round(3))

In [ ]:
plt.figure(figsize=(7, 5))

for model_name in clinical_tradeoff_df["Model"].unique():
    sub = clinical_tradeoff_df[clinical_tradeoff_df["Model"] == model_name].sort_values(
        "Dev recall constraint"
    )

    plt.plot(
        sub["Dev recall constraint"],
        sub["Test Precision"],
        marker="o",
        label=f"{model_name} precision",
    )

    plt.plot(
        sub["Dev recall constraint"],
        sub["Test Recall"],
        marker="s",
        linestyle="--",
        label=f"{model_name} recall",
    )

plt.xlabel("Minimum AD recall required on dev")
plt.ylabel("Test metric")
plt.title("Clinical threshold tradeoff on test set")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
tfidf_tradeoff = clinical_tradeoff_df[
    clinical_tradeoff_df["Model"] == "TF-IDF"
].sort_values("Dev recall constraint")

plt.figure(figsize=(6, 4))

plt.plot(
    tfidf_tradeoff["Dev recall constraint"],
    tfidf_tradeoff["Test Precision"],
    marker="o",
    label="Test precision",
)

plt.plot(
    tfidf_tradeoff["Dev recall constraint"],
    tfidf_tradeoff["Test Recall"],
    marker="s",
    label="Test recall",
)

plt.plot(
    tfidf_tradeoff["Dev recall constraint"],
    tfidf_tradeoff["Test F1"],
    marker="^",
    label="Test F1",
)

plt.xlabel("Minimum AD recall required on dev")
plt.ylabel("Test metric")
plt.title("TF-IDF clinical threshold tradeoff")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))

for model_name in clinical_tradeoff_df["Model"].unique():
    sub = clinical_tradeoff_df[clinical_tradeoff_df["Model"] == model_name].sort_values(
        "Dev recall constraint"
    )

    plt.plot(
        sub["Dev recall constraint"],
        sub["Selected threshold"],
        marker="o",
        label=model_name,
    )

plt.xlabel("Minimum AD recall required on dev")
plt.ylabel("Selected decision threshold")
plt.title("Selected threshold vs dev recall constraint")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 10. HuBERT layer sweep

The acoustic baseline uses the final hidden layer from HuBERT-base. We sweep representations from multiple HuBERT layers to test whether earlier or intermediate layers provide better acoustic features for AD detection.

In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from transformers import AutoFeatureExtractor, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hubert = AutoModel.from_pretrained(MODEL_ID)
processor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
hubert_model = hubert.to(device)
hubert_processor = processor
hubert_model.eval()


def extract_hubert_layer_features(wav_paths, layer_idx=-1, batch_size=1):
    """
    Extract mean-pooled HuBERT hidden-state features from a chosen layer.

    layer_idx:
      0  = convolutional/input projection hidden representation
      1+ = transformer layers
      -1 = final layer
    """
    feats = []

    for wav_path in tqdm(wav_paths, desc=f"Extracting HuBERT layer {layer_idx}"):
        wav, sr = torchaudio.load(wav_path)

        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        wav = wav.squeeze(0)

        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)

        inputs = hubert_processor(
            wav.numpy(),
            sampling_rate=16000,
            return_tensors="pt",
            padding=True,
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            out = hubert_model(
                **inputs,
                output_hidden_states=True,
            )

        hidden = out.hidden_states[layer_idx]  # [batch, time, hidden_dim]
        pooled = hidden.mean(dim=1).squeeze(0).detach().cpu().numpy()

        feats.append(pooled)

    return np.vstack(feats)

In [ ]:
def eval_hubert_layer(layer_idx):
    # Extract features
    Xtr = extract_hubert_layer_features(splits["train"]["wav"], layer_idx=layer_idx)
    Xdv = extract_hubert_layer_features(splits["dev"]["wav"], layer_idx=layer_idx)
    Xte = extract_hubert_layer_features(splits["test"]["wav"], layer_idx=layer_idx)

    ytr = np.array(splits["train"]["y"])
    ydv = np.array(splits["dev"]["y"])
    yte = np.array(splits["test"]["y"])

    tr_sess = np.array(splits["train"]["sess"])
    dv_sess = np.array(splits["dev"]["sess"])
    te_sess = np.array(splits["test"]["sess"])

    tr_utts = np.array(splits["train"]["utts"])
    dv_utts = np.array(splits["dev"]["utts"])
    te_utts = np.array(splits["test"]["utts"])

    # Scale features
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xdv_s = scaler.transform(Xdv)
    Xte_s = scaler.transform(Xte)

    # Match baseline style: tune C on dev AUROC
    Cs = [0.01, 0.1, 1.0, 10.0, 100.0]

    best = None

    for C in Cs:
        clf = LogisticRegression(
            C=C,
            max_iter=5000,
            class_weight="balanced",
        )

        clf.fit(Xtr_s, ytr)

        dv_seg_prob = clf.predict_proba(Xdv_s)[:, 1]
        dv_sess_prob, dv_sess_y, dv_sess_ids = aggregate_sessions(
            dv_utts,
            dv_seg_prob,
            ydv,
            dv_sess,
        )

        dv_auc = roc_auc_score(dv_sess_y, dv_sess_prob)

        if best is None or dv_auc > best["dev_AUROC"]:
            best = {
                "C": C,
                "clf": clf,
                "dev_AUROC": dv_auc,
            }

    # Evaluate best model on dev and test
    clf = best["clf"]

    rows = []

    for split_name, X_s, y, sess, utts in [
        ("dev", Xdv_s, ydv, dv_sess, dv_utts),
        ("test", Xte_s, yte, te_sess, te_utts),
    ]:
        seg_prob = clf.predict_proba(X_s)[:, 1]
        sess_prob, sess_y, sess_ids = aggregate_sessions(
            utts,
            seg_prob,
            y,
            sess,
        )

        sess_pred = (sess_prob >= 0.5).astype(int)

        rows.append({
            "layer": layer_idx,
            "split": split_name,
            "best_C": best["C"],
            "Accuracy": accuracy_score(sess_y, sess_pred),
            "F1": f1_score(sess_y, sess_pred),
            "Precision": precision_score(sess_y, sess_pred, zero_division=0),
            "Recall": recall_score(sess_y, sess_pred, zero_division=0),
            "AUROC": roc_auc_score(sess_y, sess_prob),
        })

    return pd.DataFrame(rows)

In [ ]:
layers_to_try = list(range(13))

hubert_layer_results = []

for layer_idx in layers_to_try:
    print(f"\nRunning HuBERT layer {layer_idx}")
    layer_result = eval_hubert_layer(layer_idx)
    hubert_layer_results.append(layer_result)

hubert_layer_results_df = pd.concat(hubert_layer_results, ignore_index=True)

hubert_layer_display = hubert_layer_results_df.copy()

for col in ["Accuracy", "F1", "Precision", "Recall", "AUROC"]:
    hubert_layer_display[col] = hubert_layer_display[col].map(lambda x: round(100 * x, 2))

display(hubert_layer_display)

In [ ]:
plot_df = hubert_layer_results_df[hubert_layer_results_df["split"] == "test"].copy()

plt.figure(figsize=(7, 4))
plt.plot(plot_df["layer"], 100 * plot_df["AUROC"], marker="o", label="AUROC")
plt.plot(plot_df["layer"], 100 * plot_df["F1"], marker="s", label="F1")
plt.xlabel("HuBERT layer")
plt.ylabel("Test metric")
plt.title("HuBERT layer sweep: test performance")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))

for split_name in ["dev", "test"]:
    sub = hubert_layer_results_df[hubert_layer_results_df["split"] == split_name]
    plt.plot(
        sub["layer"],
        100 * sub["AUROC"],
        marker="o",
        label=f"{split_name} AUROC",
    )

plt.xlabel("HuBERT layer")
plt.ylabel("AUROC")
plt.title("HuBERT layer sweep: dev vs test AUROC")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
def train_eval_hubert_layer_return_outputs(layer_idx):
    # Extract features
    Xtr = extract_hubert_layer_features(splits["train"]["wav"], layer_idx=layer_idx)
    Xdv = extract_hubert_layer_features(splits["dev"]["wav"], layer_idx=layer_idx)
    Xte = extract_hubert_layer_features(splits["test"]["wav"], layer_idx=layer_idx)

    ytr = np.array(splits["train"]["y"])
    ydv = np.array(splits["dev"]["y"])
    yte = np.array(splits["test"]["y"])

    tr_sess = np.array(splits["train"]["sess"])
    dv_sess = np.array(splits["dev"]["sess"])
    te_sess = np.array(splits["test"]["sess"])

    tr_utts = np.array(splits["train"]["utts"])
    dv_utts = np.array(splits["dev"]["utts"])
    te_utts = np.array(splits["test"]["utts"])

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xdv_s = scaler.transform(Xdv)
    Xte_s = scaler.transform(Xte)

    Cs = [0.01, 0.1, 1.0, 10.0, 100.0]

    best = None

    for C in Cs:
        clf = LogisticRegression(
            C=C,
            max_iter=5000,
            class_weight="balanced",
        )

        clf.fit(Xtr_s, ytr)

        dv_seg_prob = clf.predict_proba(Xdv_s)[:, 1]

        dv_sess_prob, dv_sess_y, dv_sess_ids = aggregate_sessions(
            dv_utts,
            dv_seg_prob,
            ydv,
            dv_sess,
        )

        dv_auc = roc_auc_score(dv_sess_y, dv_sess_prob)

        if best is None or dv_auc > best["dev_AUROC"]:
            best = {
                "C": C,
                "clf": clf,
                "dev_AUROC": dv_auc,
            }

    clf = best["clf"]

    # Session-level dev probabilities
    dv_seg_prob = clf.predict_proba(Xdv_s)[:, 1]
    dv_sess_prob, dv_sess_y, dv_sess_ids = aggregate_sessions(
        dv_utts,
        dv_seg_prob,
        ydv,
        dv_sess,
    )

    # Session-level test probabilities
    te_seg_prob = clf.predict_proba(Xte_s)[:, 1]
    te_sess_prob, te_sess_y, te_sess_ids = aggregate_sessions(
        te_utts,
        te_seg_prob,
        yte,
        te_sess,
    )

    dev_df = pd.DataFrame({
        "session": dv_sess_ids,
        "label": dv_sess_y,
        f"hubert_layer{layer_idx}_prob": dv_sess_prob,
    })

    test_df = pd.DataFrame({
        "session": te_sess_ids,
        "label": te_sess_y,
        f"hubert_layer{layer_idx}_prob": te_sess_prob,
    })

    return {
        "layer": layer_idx,
        "best_C": best["C"],
        "dev_AUROC": best["dev_AUROC"],
        "clf": clf,
        "scaler": scaler,
        "dev_df": dev_df,
        "test_df": test_df,
    }

In [ ]:
hubert_l9_outputs = train_eval_hubert_layer_return_outputs(layer_idx=9)

hubert_l9_dev_df = hubert_l9_outputs["dev_df"]
hubert_l9_test_df = hubert_l9_outputs["test_df"]

print("Best C for HuBERT layer 9:", hubert_l9_outputs["best_C"])
display(hubert_l9_dev_df.head())
display(hubert_l9_test_df.head())

In [ ]:
# Merge TF-IDF probabilities with HuBERT layer 9 probabilities

dev_l9_fusion_df = tfidf_dev_pred.merge(
    hubert_l9_dev_df.drop(columns=["label"]),
    on="session",
    how="inner",
)

test_l9_fusion_df = tfidf_test_pred.merge(
    hubert_l9_test_df.drop(columns=["label"]),
    on="session",
    how="inner",
)

display(dev_l9_fusion_df.head())
display(test_l9_fusion_df.head())

In [ ]:
l9_fusion_rows = []

# Individual models
for split_name, df in [("dev", dev_l9_fusion_df), ("test", test_l9_fusion_df)]:
    y = df["label"].values

    for model_name, prob_col in [
        ("TF-IDF only", "tfidf_prob"),
        ("HuBERT layer 9 only", "hubert_layer9_prob"),
    ]:
        metrics = eval_session_probs(y, df[prob_col].values)
        row = {
            "Model": model_name,
            "Split": split_name,
            "alpha_tfidf": np.nan,
        }
        row.update(metrics)
        l9_fusion_rows.append(row)


# Average fusion
for split_name, df in [("dev", dev_l9_fusion_df), ("test", test_l9_fusion_df)]:
    y = df["label"].values

    fused_prob = (
        0.5 * df["tfidf_prob"].values
        + 0.5 * df["hubert_layer9_prob"].values
    )

    metrics = eval_session_probs(y, fused_prob)

    row = {
        "Model": "TF-IDF + HuBERT L9 average fusion",
        "Split": split_name,
        "alpha_tfidf": 0.5,
    }
    row.update(metrics)
    l9_fusion_rows.append(row)


# Tune alpha on dev AUROC
alphas = np.linspace(0, 1, 21)

l9_alpha_search = []

for alpha in alphas:
    dev_y = dev_l9_fusion_df["label"].values

    dev_fused_prob = (
        alpha * dev_l9_fusion_df["tfidf_prob"].values
        + (1 - alpha) * dev_l9_fusion_df["hubert_layer9_prob"].values
    )

    metrics = eval_session_probs(dev_y, dev_fused_prob)

    l9_alpha_search.append({
        "alpha_tfidf": alpha,
        "dev_AUROC": metrics["AUROC"],
        "dev_F1": metrics["F1"],
        "dev_Accuracy": metrics["Accuracy"],
    })

l9_alpha_search_df = pd.DataFrame(l9_alpha_search)

display(l9_alpha_search_df)

best_alpha_l9 = l9_alpha_search_df.sort_values(
    ["dev_AUROC", "dev_F1"],
    ascending=False,
).iloc[0]["alpha_tfidf"]

print(f"Best alpha for TF-IDF + HuBERT layer 9 fusion: {best_alpha_l9:.2f}")

In [ ]:
# Evaluate tuned TF-IDF + HuBERT layer 9 fusion

for split_name, df in [("dev", dev_l9_fusion_df), ("test", test_l9_fusion_df)]:
    y = df["label"].values

    fused_prob = (
        best_alpha_l9 * df["tfidf_prob"].values
        + (1 - best_alpha_l9) * df["hubert_layer9_prob"].values
    )

    metrics = eval_session_probs(y, fused_prob)

    row = {
        "Model": "TF-IDF + HuBERT L9 weighted fusion",
        "Split": split_name,
        "alpha_tfidf": best_alpha_l9,
    }
    row.update(metrics)
    l9_fusion_rows.append(row)


l9_fusion_results = pd.DataFrame(l9_fusion_rows)

l9_fusion_display = l9_fusion_results.copy()

for col in ["Accuracy", "F1", "Precision", "Recall", "AUROC"]:
    l9_fusion_display[col] = l9_fusion_display[col].map(lambda x: round(100 * x, 2))

l9_fusion_display["alpha_tfidf"] = l9_fusion_display["alpha_tfidf"].map(
    lambda x: "" if pd.isna(x) else round(float(x), 2)
)

display(l9_fusion_display)

In [ ]:
fusion_comparison_rows = []

# Existing final-layer weighted fusion
final_layer_test = fusion_results[
    (fusion_results["Model"] == "Weighted fusion tuned on dev")
    & (fusion_results["Split"] == "test")
].copy()

# New layer-9 weighted fusion
layer9_test = l9_fusion_results[
    (l9_fusion_results["Model"] == "TF-IDF + HuBERT L9 weighted fusion")
    & (l9_fusion_results["Split"] == "test")
].copy()

# TF-IDF only
tfidf_test = l9_fusion_results[
    (l9_fusion_results["Model"] == "TF-IDF only")
    & (l9_fusion_results["Split"] == "test")
].copy()

comparison = pd.concat(
    [
        tfidf_test.assign(Comparison_Model="TF-IDF only"),
        final_layer_test.assign(Comparison_Model="TF-IDF + HuBERT final layer fusion"),
        layer9_test.assign(Comparison_Model="TF-IDF + HuBERT layer 9 fusion"),
    ],
    ignore_index=True,
)

comparison_display = comparison[
    [
        "Comparison_Model",
        "alpha_tfidf",
        "Accuracy",
        "F1",
        "Precision",
        "Recall",
        "AUROC",
    ]
].copy()

for col in ["Accuracy", "F1", "Precision", "Recall", "AUROC"]:
    comparison_display[col] = comparison_display[col].map(lambda x: round(100 * x, 2))

comparison_display["alpha_tfidf"] = comparison_display["alpha_tfidf"].map(
    lambda x: "" if pd.isna(x) else round(float(x), 2)
)

display(comparison_display)